In [ ]:
!pip install pandas duckdb pytest faker numpy

### 1.Project Setup and Library Imports

In [1]:
import pandas as pd
import numpy as np
import duckdb
import faker
import os
import logging
import uuid
import time
from datetime import datetime

print("All libraries imported successfully!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("DuckDB version:", duckdb.__version__)

All libraries imported successfully!
Pandas version: 3.0.1
NumPy version: 2.4.2
DuckDB version: 2.0.0


### Create project directories

In [2]:
folders = [
    "data",
    "output",
    "logs",
    "tests",
    "sql",
    "data/quarantine"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [3]:
for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d != ".ipynb_checkpoints"]
    print(root)
    for file in files:
        print("   ", file)

.
    Untitled.ipynb
.\.pytest_cache
    .gitignore
    CACHEDIR.TAG
    README.md
.\.pytest_cache\v
.\.pytest_cache\v\cache
    nodeids
.\data
    customers.csv
    orders.csv
    order_items.csv
    products.csv
.\data\quarantine
.\logs
    pipeline.log
    pipeline_audit.csv
    pipeline_run_summary.csv
.\output
    category_sales.csv
    dim_customer.csv
    dim_date.csv
    dim_product.csv
    ecommerce_warehouse.duckdb
    fact_sales.csv
    ingestion_summary.csv
    monthly_sales.csv
    regional_sales.csv
    test_results.txt
    top_customers.csv
    top_products.csv
.\sql
.\src
    validation.py
.\src\__pycache__
    validation.cpython-313.pyc
.\tests
    test_pipeline.py
.\tests\__pycache__
    test_pipeline.cpython-313-pytest-9.2.0.pyc


## Generate sample datasets

In [4]:
from faker import Faker
import random
import pandas as pd
import numpy as np
import os

fake = Faker()

# Fixed random seed for reproducibility
Faker.seed(42)
random.seed(42)
np.random.seed(42)

print("Dataset generation environment ready!")

Dataset generation environment ready!


### Generate customers dataset

In [5]:
customers = []

for i in range(1, 101):
    customers.append({
        "customer_id": f"CUST{i:04d}",
        "customer_name": fake.name(),
        "email": fake.email(),
        "city": random.choice([
            "Hyderabad", "Bangalore", "Chennai",
            "Mumbai", "Delhi", "Pune"
        ]),
        "region": random.choice([
            "South", "North", "West", "East"
        ]),
        "signup_date": fake.date_between(
            start_date="-3y",
            end_date="today"
        )
    })

customers_df = pd.DataFrame(customers)

customers_df.to_csv(
    "data/customers.csv", index=False
)

print("Customers generated:", len(customers_df))

customers_df.head()

Customers generated: 100


,customer_id,customer_name,email,city,region,signup_date
0,CUST0001,Allison Hill,donaldgarcia@example.net,Pune,South,2026-03-02
1,CUST0002,Javier Johnson,jesseguzman@example.net,Hyderabad,West,2024-09-09
2,CUST0003,Meredith Barnes,zlawrence@example.org,Bangalore,North,2024-07-11
3,CUST0004,Gina Moore,curtis61@example.com,Bangalore,South,2025-04-23
4,CUST0005,Renee Blair,dudleynicholas@example.net,Pune,South,2025-02-03


### Generate products datase

In [6]:
products = []

categories = [
    "Electronics",
    "Clothing",
    "Furniture",
    "Books",
    "Accessories"
]

for i in range(1, 31):
    products.append({
        "product_id": f"PROD{i:03d}",
        "product_name": fake.word().capitalize(),
        "category": random.choice(categories),
        "unit_price": round(random.uniform(100, 50000), 2)
    })

products_df = pd.DataFrame(products)

products_df.to_csv(
    "data/products.csv", index=False
)

print("Products generated:", len(products_df))

products_df.head()

Products generated: 30


,product_id,product_name,category,unit_price
0,PROD001,You,Electronics,11524.20
1,PROD002,Institution,Electronics,42995.81
2,PROD003,Teach,Electronics,25756.39
3,PROD004,Soon,Furniture,33481.99
4,PROD005,Oil,Clothing,27007.77


## Generate orders dataset

In [7]:
orders = []

for i in range(1, 501):
    orders.append({
        "order_id": f"ORD{i:05d}",
        "customer_id": random.choice(
            customers_df["customer_id"].tolist()
        ),
        "order_date": fake.date_between(
            start_date="-1y",
            end_date="today"
        ),
        "order_status": random.choice([
            "Completed", "Pending", "Cancelled"
        ])
    })

orders_df = pd.DataFrame(orders)

orders_df.to_csv(
    "data/orders.csv", index=False
)

print("Orders generated:", len(orders_df))

orders_df.head()

Orders generated: 500


,order_id,customer_id,order_date,order_status
0,ORD00001,CUST0038,2026-07-05,Completed
1,ORD00002,CUST0008,2026-04-25,Cancelled
2,ORD00003,CUST0095,2026-08-27,Cancelled
3,ORD00004,CUST0008,2025-11-25,Cancelled
4,ORD00005,CUST0041,2026-01-23,Completed


### Generate order items dataset

In [8]:
order_items = []

for i in range(1, 1001):
    product = products_df.sample(1).iloc[0]

    order_items.append({
        "order_item_id": f"ITEM{i:05d}",
        "order_id": random.choice(
            orders_df["order_id"].tolist()
        ),
        "product_id": product["product_id"],
        "quantity": random.randint(1, 5),
        "unit_price": product["unit_price"],
        "discount": round(random.uniform(0, 100), 2)
    })

order_items_df = pd.DataFrame(order_items)

order_items_df.to_csv(
    "data/order_items.csv", index=False
)

print("Order items generated:", len(order_items_df))

order_items_df.head()

Order items generated: 1000


,order_item_id,order_id,product_id,quantity,unit_price,discount
0,ITEM00001,ORD00318,PROD028,2,14333.94,49.77
1,ITEM00002,ORD00299,PROD001,2,11524.20,23.22
2,ITEM00003,ORD00327,PROD005,3,27007.77,45.96
3,ITEM00004,ORD00342,PROD004,1,33481.99,89.89
4,ITEM00005,ORD00239,PROD027,3,13333.63,67.76


### Verify generated files

In [9]:
files = [
    "data/customers.csv",
    "data/products.csv",
    "data/orders.csv",
    "data/order_items.csv"
]

for file in files:
    df = pd.read_csv(file)

    print(
        file,
        "| Rows:", len(df),
        "| Columns:", len(df.columns)
    )

data/customers.csv | Rows: 100 | Columns: 6
data/products.csv | Rows: 30 | Columns: 4
data/orders.csv | Rows: 500 | Columns: 4
data/order_items.csv | Rows: 1000 | Columns: 6


# Step 3 – Introduce data quality issue

## 3.1 Understand what we are going to introduce

### Data Quality Issue Generation

In [10]:
# Create copies of original datasets
customers_bad = customers_df.copy()
products_bad = products_df.copy()
orders_bad = orders_df.copy()
order_items_bad = order_items_df.copy()

# 1. Missing customer name
customers_bad.loc[0, "customer_name"] = None

# 2. Missing email
customers_bad.loc[1, "email"] = None

# 3. Unexpected product category
products_bad.loc[0, "category"] = "Unknown_Category"

# 4. Invalid customer foreign key
orders_bad.loc[0, "customer_id"] = "CUST9999"

# 5. Invalid date
orders_bad.loc[1, "order_date"] = "INVALID_DATE"

# 6. Negative quantity
order_items_bad.loc[0, "quantity"] = -5

# 7. Missing product ID
order_items_bad.loc[1, "product_id"] = None

# 8. Duplicate order item
duplicate_record = order_items_bad.iloc[[2]].copy()

order_items_bad = pd.concat(
    [order_items_bad, duplicate_record],
    ignore_index=True
)

# 9. Invalid product foreign key
order_items_bad.loc[3, "product_id"] = "PROD999"

print("Data quality issues introduced successfully!")

Data quality issues introduced successfully!


### Save the datasets with data-quality issues

In [11]:
customers_bad.to_csv("data/customers.csv", index=False)

products_bad.to_csv("data/products.csv", index=False)

orders_bad.to_csv("data/orders.csv", index=False)

order_items_bad.to_csv("data/order_items.csv", index=False)

print("Updated datasets saved successfully!")

Updated datasets saved successfully!


### Verify the issues

In [12]:
print("Missing customer names:",
      customers_bad["customer_name"].isnull().sum())

print("Missing customer emails:",
      customers_bad["email"].isnull().sum())

print("Invalid customer IDs:",
      (~orders_bad["customer_id"].isin(
          customers_bad["customer_id"]
      )).sum())

print("Negative quantities:",
      (order_items_bad["quantity"] < 0).sum())

print("Missing product IDs:",
      order_items_bad["product_id"].isnull().sum())

print("Total order item records:",
      len(order_items_bad))

print("Duplicate order item IDs:",
      order_items_bad["order_item_id"].duplicated().sum())

Missing customer names: 1
Missing customer emails: 1
Invalid customer IDs: 1
Negative quantities: 1
Missing product IDs: 1
Total order item records: 1001
Duplicate order item IDs: 1


# Step 4 – Data Ingestion

### Load source CSV files

In [13]:
# Load source datasets

customers_raw = pd.read_csv("data/customers.csv")

products_raw = pd.read_csv("data/products.csv")

orders_raw = pd.read_csv("data/orders.csv")

order_items_raw = pd.read_csv("data/order_items.csv")

print("All source datasets loaded successfully!")

All source datasets loaded successfully!


### Check dataset sizes

In [14]:
source_datasets = {
    "customers": customers_raw,
    "products": products_raw,
    "orders": orders_raw,
    "order_items": order_items_raw
}

for name, df in source_datasets.items():
    print(f"{name}: {len(df)} records")

customers: 100 records
products: 30 records
orders: 500 records
order_items: 1001 records


### Inspect the data types

In [15]:
for name, df in source_datasets.items():
    print("\nDataset:", name)
    print(df.dtypes)


Dataset: customers
customer_id      str
customer_name    str
email            str
city             str
region           str
signup_date      str
dtype: object

Dataset: products
product_id          str
product_name        str
category            str
unit_price      float64
dtype: object

Dataset: orders
order_id        str
customer_id     str
order_date      str
order_status    str
dtype: object

Dataset: order_items
order_item_id        str
order_id             str
product_id           str
quantity           int64
unit_price       float64
discount         float64
dtype: object


### Check column names

In [16]:
for name, df in source_datasets.items():
    print(f"\n{name} columns:")
    print(df.columns.tolist())


customers columns:
['customer_id', 'customer_name', 'email', 'city', 'region', 'signup_date']

products columns:
['product_id', 'product_name', 'category', 'unit_price']

orders columns:
['order_id', 'customer_id', 'order_date', 'order_status']

order_items columns:
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'discount']


### Generate ingestion summary

In [17]:
ingestion_summary = pd.DataFrame([
    {
        "source_file": f"{name}.csv",
        "total_records": len(df),
        "total_columns": len(df.columns),
        "missing_values": df.isnull().sum().sum()
    }
    for name, df in source_datasets.items()
])

display(ingestion_summary)

,source_file,total_records,total_columns,missing_values
0,customers.csv,100,6,2
1,products.csv,30,4,0
2,orders.csv,500,4,0
3,order_items.csv,1001,6,1


In [18]:
ingestion_summary.to_csv(
    "output/ingestion_summary.csv",
    index=False
)

print("Ingestion summary saved successfully!")

Ingestion summary saved successfully!


# Step 5 – Data validation

### What will we validate?

### Customer validation

In [19]:

customers_valid = customers_raw.copy()

customers_rejected = customers_valid[
    customers_valid["customer_name"].isnull() |
    customers_valid["email"].isnull() |
    customers_valid["customer_id"].duplicated(keep=False)
].copy()

customers_valid = customers_valid.dropna(
    subset=["customer_name", "email"]
)

customers_valid = customers_valid.drop_duplicates(
    subset=["customer_id"],
    keep="first"
)

print("Valid customers:", len(customers_valid))
print("Rejected customers:", len(customers_rejected))

Valid customers: 98
Rejected customers: 2


### Validate products

In [20]:
# Product validation

valid_categories = [
    "Electronics",
    "Clothing",
    "Furniture",
    "Books",
    "Accessories"
]

products_valid = products_raw.copy()

invalid_products = (
    ~products_valid["category"].isin(valid_categories)
)

duplicate_products = products_valid[
    "product_id"
].duplicated(keep=False)

products_rejected = products_valid[
    invalid_products | duplicate_products
].copy()

products_valid = products_valid[
    ~invalid_products
].drop_duplicates(
    subset=["product_id"],
    keep="first"
)

print("Valid products:", len(products_valid))
print("Rejected products:", len(products_rejected))

Valid products: 29
Rejected products: 1


### Validate orders

In [21]:
# Order validation

orders_check = orders_raw.copy()

orders_check["order_date"] = pd.to_datetime(
    orders_check["order_date"],
    errors="coerce"
)

invalid_customer = ~orders_check["customer_id"].isin(
    customers_valid["customer_id"]
)

invalid_date = orders_check["order_date"].isnull()

duplicate_order = orders_check["order_id"].duplicated(
    keep=False
)

orders_rejected = orders_check[
    invalid_customer | invalid_date | duplicate_order
].copy()

orders_valid = orders_check[
    ~(invalid_customer | invalid_date | duplicate_order)
].copy()

print("Valid orders:", len(orders_valid))
print("Rejected orders:", len(orders_rejected))

Valid orders: 492
Rejected orders: 8


### Validate order items

In [22]:
# Order item validation

items_check = order_items_raw.copy()

invalid_order = ~items_check["order_id"].isin(
    orders_valid["order_id"]
)

invalid_product = (
    items_check["product_id"].isna() |
    ~items_check["product_id"].isin(
        products_valid["product_id"]
    )
)

invalid_quantity = (
    items_check["quantity"].isna() |
    (items_check["quantity"] <= 0)
)

invalid_price = (
    items_check["unit_price"].isna() |
    (items_check["unit_price"] <= 0)
)

invalid_discount = (
    items_check["discount"].isna() |
    (items_check["discount"] < 0) |
    (items_check["discount"] > items_check["quantity"] * items_check["unit_price"])
)

duplicate_item = items_check[
    "order_item_id"
].duplicated(keep=False)

invalid_items = (
    invalid_order |
    invalid_product |
    invalid_quantity |
    invalid_price |
    invalid_discount |
    duplicate_item
)

order_items_rejected = items_check[invalid_items].copy()

order_items_valid = items_check[~invalid_items].copy()

print("Valid order items:", len(order_items_valid))
print("Rejected order items:", len(order_items_rejected))

Valid order items: 954
Rejected order items: 47


# Step 6 – Data transformation

## Understand what we are building

### Important design decisions:

### Customer Dimension

In [23]:
# Create customer dimension

dim_customer = customers_valid[
    ["customer_id", "customer_name", "email", "city", "region"]
].copy()

# Generate deterministic surrogate keys
dim_customer = dim_customer.sort_values("customer_id").reset_index(drop=True)

dim_customer.insert(
    0,
    "customer_key",
    range(1, len(dim_customer) + 1)
)

print("Customer dimension created:", len(dim_customer))

display(dim_customer.head())

Customer dimension created: 98


,customer_key,customer_id,customer_name,email,city,region
0,1,CUST0003,Meredith Barnes,zlawrence@example.org,Bangalore,North
1,2,CUST0004,Gina Moore,curtis61@example.com,Bangalore,South
2,3,CUST0005,Renee Blair,dudleynicholas@example.net,Pune,South
3,4,CUST0006,Lisa Hensley,janetwilliams@example.org,Delhi,East
4,5,CUST0007,Dr. Sharon James,xreid@example.org,Hyderabad,South


### Product Dimension

In [24]:
# Create product dimension

dim_product = products_valid[
    ["product_id", "product_name", "category", "unit_price"]
].copy()

dim_product = dim_product.sort_values("product_id").reset_index(drop=True)

dim_product.insert(
    0,
    "product_key",
    range(1, len(dim_product) + 1)
)

print("Product dimension created:", len(dim_product))

display(dim_product.head())

Product dimension created: 29


,product_key,product_id,product_name,category,unit_price
0,1,PROD002,Institution,Electronics,42995.81
1,2,PROD003,Teach,Electronics,25756.39
2,3,PROD004,Soon,Furniture,33481.99
3,4,PROD005,Oil,Clothing,27007.77
4,5,PROD006,Although,Accessories,28852.96


### Date Dimension

In [25]:
# Consider completed orders only
completed_orders = orders_valid[
    orders_valid["order_status"] == "Completed"
].copy()

# Extract unique order dates
date_values = sorted(
    completed_orders["order_date"].dropna().unique()
)

dim_date = pd.DataFrame({
    "full_date": pd.to_datetime(date_values)
})

dim_date["date_key"] = (
    dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
)

dim_date["day"] = dim_date["full_date"].dt.day
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["year"] = dim_date["full_date"].dt.year

dim_date = dim_date[
    ["date_key", "full_date", "day", "month",
     "month_name", "quarter", "year"]
]

print("Date dimension created:", len(dim_date))

display(dim_date.head())

Date dimension created: 132


,date_key,full_date,day,month,month_name,quarter,year
0,20251105,2025-11-05,5,11,November,4,2025
1,20251106,2025-11-06,6,11,November,4,2025
2,20251113,2025-11-13,13,11,November,4,2025
3,20251114,2025-11-14,14,11,November,4,2025
4,20251116,2025-11-16,16,11,November,4,2025


### Fact Sales Table

In [26]:
# Select order items belonging to completed orders

sales_items = order_items_valid.merge(
    completed_orders[
        ["order_id", "customer_id", "order_date"]
    ],
    on="order_id",
    how="inner",
    validate="many_to_one"
)

# Join customer surrogate keys
fact_sales = sales_items.merge(
    dim_customer[["customer_id", "customer_key"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Join product surrogate keys
fact_sales = fact_sales.merge(
    dim_product[["product_id", "product_key"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

# Generate date key
fact_sales["date_key"] = (
    pd.to_datetime(fact_sales["order_date"])
    .dt.strftime("%Y%m%d")
    .astype(int)
)

# Calculate total sales
fact_sales["gross_amount"] = (
    fact_sales["quantity"] * fact_sales["unit_price"]
)

fact_sales["sales_amount"] = (
    fact_sales["gross_amount"] - fact_sales["discount"]
)

# Select final fact table columns
fact_sales = fact_sales[
    [
        "order_item_id",
        "order_id",
        "customer_key",
        "product_key",
        "date_key",
        "quantity",
        "unit_price",
        "discount",
        "gross_amount",
        "sales_amount"
    ]
].copy()

print("Fact sales table created:", len(fact_sales))

display(fact_sales.head())

Fact sales table created: 323


,order_item_id,order_id,customer_key,product_key,date_key,quantity,unit_price,discount,gross_amount,sales_amount
0,ITEM00014,ORD00290,96,12,20260117,1,3124.57,91.06,3124.57,3033.51
1,ITEM00015,ORD00382,55,8,20251114,4,32983.24,83.28,131932.96,131849.68
2,ITEM00016,ORD00463,82,23,20260624,3,24119.79,77.57,72359.37,72281.80
3,ITEM00020,ORD00463,82,12,20260624,1,3124.57,30.89,3124.57,3093.68
4,ITEM00024,ORD00090,28,20,20260607,2,41817.78,13.22,83635.56,83622.34


### Verify dimensional model

In [27]:
# Primary key checks

assert dim_customer["customer_key"].is_unique
assert dim_product["product_key"].is_unique
assert dim_date["date_key"].is_unique
assert fact_sales["order_item_id"].is_unique

# Foreign key checks

assert fact_sales["customer_key"].isin(
    dim_customer["customer_key"]
).all()

assert fact_sales["product_key"].isin(
    dim_product["product_key"]
).all()

assert fact_sales["date_key"].isin(
    dim_date["date_key"]
).all()

print("All primary key checks passed!")
print("All foreign key checks passed!")
print("Dimensional model validation successful!")

All primary key checks passed!
All foreign key checks passed!
Dimensional model validation successful!


### Display final table counts

In [28]:
warehouse_summary = pd.DataFrame({
    "table_name": [
        "dim_customer",
        "dim_product",
        "dim_date",
        "fact_sales"
    ],
    "record_count": [
        len(dim_customer),
        len(dim_product),
        len(dim_date),
        len(fact_sales)
    ]
})

display(warehouse_summary)

,table_name,record_count
0,dim_customer,98
1,dim_product,29
2,dim_date,132
3,fact_sales,323


### Save dimensional tables

In [29]:
dim_customer.to_csv(
    "output/dim_customer.csv", index=False
)

dim_product.to_csv(
    "output/dim_product.csv", index=False
)

dim_date.to_csv(
    "output/dim_date.csv", index=False
)

fact_sales.to_csv(
    "output/fact_sales.csv", index=False
)

print("All dimensional tables saved successfully!")

All dimensional tables saved successfully!


# Step 7 – Create the DuckDB Data Warehouse

## What is DuckDB?

### Connect to DuckDB

In [45]:
import duckdb
import os

# Create database connection
con = duckdb.connect("output/ecommerce_warehouse.duckdb")

print("DuckDB connection successful!")

DuckDB connection successful!


### Create customer dimension table

In [99]:
con.execute("""
CREATE OR REPLACE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY,
    customer_id VARCHAR UNIQUE NOT NULL,
    customer_name VARCHAR NOT NULL,
    email VARCHAR NOT NULL,
    city VARCHAR,
    region VARCHAR
)
""")

con.register("customer_df", dim_customer)

con.execute("""
INSERT INTO dim_customer
SELECT * FROM customer_df
""")

print("Customer dimension loaded!")

CatalogException: Catalog Error: Could not drop the table because this table is main key table of the table "fact_sales"

### Create product dimension table

In [100]:
con.execute("""
CREATE OR REPLACE TABLE dim_product (
    product_key INTEGER PRIMARY KEY,
    product_id VARCHAR UNIQUE NOT NULL,
    product_name VARCHAR NOT NULL,
    category VARCHAR NOT NULL,
    unit_price DOUBLE NOT NULL
)
""")

con.register("product_df", dim_product)

con.execute("""
INSERT INTO dim_product
SELECT * FROM product_df
""")

print("Product dimension loaded!")

CatalogException: Catalog Error: Could not drop the table because this table is main key table of the table "fact_sales"

### Create date dimension table

In [101]:
con.execute("""
CREATE OR REPLACE TABLE dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date DATE NOT NULL,
    day INTEGER,
    month INTEGER,
    month_name VARCHAR,
    quarter INTEGER,
    year INTEGER
)
""")

con.register("date_df", dim_date)

con.execute("""
INSERT INTO dim_date
SELECT * FROM date_df
""")

print("Date dimension loaded!")

CatalogException: Catalog Error: Could not drop the table because this table is main key table of the table "fact_sales"

### Create fact sales table

In [102]:
con.execute("""
CREATE TABLE IF NOT EXISTS fact_sales (
    order_item_id VARCHAR PRIMARY KEY,
    order_id VARCHAR NOT NULL,
    customer_key INTEGER NOT NULL,
    product_key INTEGER NOT NULL,
    date_key INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price DOUBLE NOT NULL,
    discount DOUBLE NOT NULL,
    gross_amount DOUBLE NOT NULL,
    sales_amount DOUBLE NOT NULL,

    FOREIGN KEY (customer_key)
        REFERENCES dim_customer(customer_key),

    FOREIGN KEY (product_key)
        REFERENCES dim_product(product_key),

    FOREIGN KEY (date_key)
        REFERENCES dim_date(date_key)
)
""")

con.register("fact_df", fact_sales)

con.execute("""
INSERT INTO fact_sales
SELECT * FROM fact_df
""")

print("Fact sales table loaded!")

ConstraintException: Constraint Error: Duplicate key "order_item_id: ITEM00014" violates primary key constraint.

In [50]:
tables = con.execute("""
SHOW TABLES
""").df()

display(tables)

,name
0,dim_customer
1,dim_date
2,dim_product
3,fact_df
4,fact_sales
5,pipeline_audit


### check row counts

In [51]:
for table in [
    "dim_customer",
    "dim_product",
    "dim_date",
    "fact_sales"
]:
    count = con.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"{table}: {count} records")

dim_customer: 98 records
dim_product: 29 records
dim_date: 125 records
fact_sales: 299 records


### Verify database schema

In [52]:
for table in [
    "dim_customer",
    "dim_product",
    "dim_date",
    "fact_sales"
]:
    print("\nTable:", table)

    schema = con.execute(
        f"DESCRIBE {table}"
    ).df()

    display(schema)


Table: dim_customer


,column_name,column_type,null,key,default,extra
0,customer_key,INTEGER,NO,PRI,None,None
1,customer_id,VARCHAR,NO,UNI,None,None
2,customer_name,VARCHAR,NO,NaN,None,None
3,email,VARCHAR,NO,NaN,None,None
4,city,VARCHAR,YES,NaN,None,None
5,region,VARCHAR,YES,NaN,None,None



Table: dim_product


,column_name,column_type,null,key,default,extra
0,product_key,INTEGER,NO,PRI,None,None
1,product_id,VARCHAR,NO,UNI,None,None
2,product_name,VARCHAR,NO,NaN,None,None
3,category,VARCHAR,NO,NaN,None,None
4,unit_price,DOUBLE,NO,NaN,None,None



Table: dim_date


,column_name,column_type,null,key,default,extra
0,date_key,INTEGER,NO,PRI,None,None
1,full_date,DATE,NO,NaN,None,None
2,day,INTEGER,YES,NaN,None,None
3,month,INTEGER,YES,NaN,None,None
4,month_name,VARCHAR,YES,NaN,None,None
5,quarter,INTEGER,YES,NaN,None,None
6,year,INTEGER,YES,NaN,None,None



Table: fact_sales


,column_name,column_type,null,key,default,extra
0,order_item_id,VARCHAR,NO,PRI,None,None
1,order_id,VARCHAR,NO,NaN,None,None
2,customer_key,INTEGER,NO,NaN,None,None
3,product_key,INTEGER,NO,NaN,None,None
4,date_key,INTEGER,NO,NaN,None,None
5,quantity,INTEGER,NO,NaN,None,None
6,unit_price,DOUBLE,NO,NaN,None,None
7,discount,DOUBLE,NO,NaN,None,None
8,gross_amount,DOUBLE,NO,NaN,None,None
9,sales_amount,DOUBLE,NO,NaN,None,None


## Verify foreign key relationships

In [53]:
result = con.execute("""
SELECT
    COUNT(*) AS total_sales,
    COUNT(DISTINCT customer_key) AS unique_customers,
    COUNT(DISTINCT product_key) AS unique_products,
    ROUND(SUM(sales_amount), 2) AS total_revenue
FROM fact_sales
""").df()

display(result)

,total_sales,unique_customers,unique_products,total_revenue
0,299,76,29,20385497.29


In [54]:
dim_customer.to_csv("output/dim_customer.csv", index=False)

dim_product.to_csv("output/dim_product.csv", index=False)

dim_date.to_csv("output/dim_date.csv", index=False)

fact_sales.to_csv("output/fact_sales.csv", index=False)

print("All four CSV files generated successfully!")

All four CSV files generated successfully!


# Step 8 – SQL Analytical Queries

### 1. Find the total revenue generated by the e-commerce company.

In [55]:
total_revenue = con.execute("""
SELECT
    ROUND(SUM(sales_amount), 2) AS total_revenue
FROM fact_sales
""").df()

display(total_revenue)

,total_revenue
0,20385497.29


### 2.Find monthly revenue and sales quantity.

In [56]:
monthly_sales = con.execute("""
SELECT
    d.year,
    d.month,
    d.month_name,
    ROUND(SUM(f.sales_amount), 2) AS monthly_revenue,
    SUM(f.quantity) AS total_quantity
FROM fact_sales f
JOIN dim_date d
    ON f.date_key = d.date_key
GROUP BY
    d.year,
    d.month,
    d.month_name
ORDER BY
    d.year,
    d.month
""").df()

display(monthly_sales)

,year,month,month_name,monthly_revenue,total_quantity
0,2025,11,November,1423460.56,55.0
1,2025,12,December,1155524.22,47.0
2,2026,1,January,1280082.03,43.0
3,2026,2,February,2021152.36,76.0
4,2026,3,March,1340343.77,66.0
5,2026,4,April,2603752.63,118.0
6,2026,5,May,2021199.90,100.0
7,2026,6,June,1462484.96,71.0
8,2026,7,July,1901259.86,74.0
9,2026,8,August,1335125.33,52.0


### 3.Identify the five products generating the highest revenue

In [57]:
top_products = con.execute("""
SELECT
    p.product_name,
    p.category,
    SUM(f.quantity) AS units_sold,
    ROUND(SUM(f.sales_amount), 2) AS total_revenue
FROM fact_sales f
JOIN dim_product p
    ON f.product_key = p.product_key
GROUP BY
    p.product_name,
    p.category
ORDER BY total_revenue DESC
LIMIT 5
""").df()

display(top_products)

,product_name,category,units_sold,total_revenue
0,Pick,Furniture,44.0,1379909.61
1,Able,Books,31.0,1352035.52
2,Agree,Accessories,38.0,1296159.81
3,Direction,Electronics,29.0,1267636.71
4,Check,Furniture,23.0,1146709.67


### 4.Revenue by product category

In [58]:
category_sales = con.execute("""
SELECT
    p.category,
    ROUND(SUM(f.sales_amount), 2) AS total_revenue,
    SUM(f.quantity) AS total_quantity
FROM fact_sales f
JOIN dim_product p
    ON f.product_key = p.product_key
GROUP BY p.category
ORDER BY total_revenue DESC
""").df()

display(category_sales)

,category,total_revenue,total_quantity
0,Furniture,5874068.07,196.0
1,Electronics,5455897.18,243.0
2,Books,4121178.79,217.0
3,Clothing,2900828.87,147.0
4,Accessories,2033524.38,81.0


### 5.Identify your highest-value customers.

In [59]:
top_customers = con.execute("""
SELECT
    c.customer_id,
    c.customer_name,
    c.city,
    ROUND(SUM(f.sales_amount), 2) AS total_spending
FROM fact_sales f
JOIN dim_customer c
    ON f.customer_key = c.customer_key
GROUP BY
    c.customer_id,
    c.customer_name,
    c.city
ORDER BY total_spending DESC
LIMIT 10
""").df()

display(top_customers)

,customer_id,customer_name,city,total_spending
0,CUST0066,Tiffany Barnes,Delhi,839814.23
1,CUST0090,Bailey Duran DDS,Chennai,777906.54
2,CUST0011,Jonathan Wilkerson,Pune,666704.95
3,CUST0071,Marissa Webster,Mumbai,616870.49
4,CUST0029,John Hancock,Hyderabad,563052.85
5,CUST0008,Michael Miles,Hyderabad,555637.99
6,CUST0006,Lisa Hensley,Delhi,519795.89
7,CUST0033,Steve Sanchez,Mumbai,509352.60
8,CUST0073,Tracey Higgins,Pune,480576.81
9,CUST0019,Rose Spence,Hyderabad,470988.41


### 6.Orders by region

In [60]:
regional_sales = con.execute("""
SELECT
    c.region,
    COUNT(DISTINCT f.order_id) AS total_orders,
    ROUND(SUM(f.sales_amount), 2) AS total_revenue
FROM fact_sales f
JOIN dim_customer c
    ON f.customer_key = c.customer_key
GROUP BY c.region
ORDER BY total_revenue DESC
""").df()

display(regional_sales)

,region,total_orders,total_revenue
0,North,40,5802679.48
1,East,33,5382912.75
2,West,32,4867779.66
3,South,31,4332125.40


### 7.Calculate average order value

In [61]:
average_order_value = con.execute("""
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(sales_amount), 2) AS total_revenue,
    ROUND(
        SUM(sales_amount) /
        NULLIF(COUNT(DISTINCT order_id), 0),
        2
    ) AS average_order_value
FROM fact_sales
""").df()

display(average_order_value)

,total_orders,total_revenue,average_order_value
0,136,20385497.29,149893.36


### Verify business metrics

In [62]:
business_summary = con.execute("""
SELECT
    COUNT(*) AS total_order_items,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_key) AS unique_customers,
    COUNT(DISTINCT product_key) AS unique_products,
    SUM(quantity) AS units_sold,
    ROUND(SUM(sales_amount), 2) AS total_revenue
FROM fact_sales
""").df()

display(business_summary)

,total_order_items,total_orders,unique_customers,unique_products,units_sold,total_revenue
0,299,136,76,29,884.0,20385497.29


### Save analytical reports

In [63]:
monthly_sales.to_csv("output/monthly_sales.csv", index=False)
top_products.to_csv("output/top_products.csv", index=False)
category_sales.to_csv("output/category_sales.csv", index=False)
top_customers.to_csv("output/top_customers.csv", index=False)
regional_sales.to_csv("output/regional_sales.csv", index=False)

print("All analytical reports saved successfully!")

All analytical reports saved successfully!


# Step 9 – Pipeline monitoring and logging

### What are we going to track?

### Import monitoring libraries

In [64]:
import logging
import uuid
import time
import os
import pandas as pd
from datetime import datetime, timezone

### Configure logging

In [65]:
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

print("Logging configured successfully!")

Logging configured successfully!


### Create a pipeline run ID

In [66]:
run_id = str(uuid.uuid4())

start_time = datetime.now(timezone.utc)

print("Pipeline Run ID:", run_id)
print("Start Time:", start_time.isoformat())

logging.info(f"Pipeline started. Run ID: {run_id}")

Pipeline Run ID: 7a5106b0-7686-4993-86b9-36d701ee36fa
Start Time: 2026-11-06T07:52:52.610244+00:00


### Capture input and output counts

In [67]:
input_records = (
    len(customers_raw)
    + len(products_raw)
    + len(orders_raw)
    + len(order_items_raw)
)

output_records = (
    len(dim_customer)
    + len(dim_product)
    + len(dim_date)
    + len(fact_sales)
)

rejected_records = (
    len(customers_rejected)
    + len(products_rejected)
    + len(orders_rejected)
    + len(order_items_rejected)
)

print("Input records:", input_records)
print("Output records:", output_records)
print("Rejected records:", rejected_records)

Input records: 1631
Output records: 582
Rejected records: 58


### Create pipeline execution summary

In [68]:
end_time = datetime.now(timezone.utc)

execution_seconds = (
    end_time - start_time
).total_seconds()

pipeline_status = "SUCCESS"

run_summary = pd.DataFrame([{
    "run_id": run_id,
    "start_time": start_time.isoformat(),
    "end_time": end_time.isoformat(),
    "input_records": input_records,
    "output_records": output_records,
    "rejected_records": rejected_records,
    "execution_seconds": execution_seconds,
    "status": pipeline_status,
    "failure_reason": None
}])

display(run_summary)

,run_id,start_time,end_time,input_records,output_records,rejected_records,execution_seconds,status,failure_reason
0,7a5106b0-7686-4993-86b9-36d701ee36fa,2026-11-06T07:52:52.610244+00:00,2026-11-06T07:52:53.325675+00:00,1631,582,58,0.715431,SUCCESS,None


### Save pipeline monitoring report

In [69]:
run_summary.to_csv(
    "logs/pipeline_run_summary.csv",
    mode="a",
    header=not os.path.exists(
        "logs/pipeline_run_summary.csv"
    ),
    index=False
)

logging.info(
    f"Pipeline completed successfully. Run ID: {run_id}"
)

print("Pipeline monitoring report saved!")

Pipeline monitoring report saved!


### Verify the log file

In [70]:
with open("logs/pipeline.log", "r") as file:
    print(file.read())

2026-11-06 13:00:09,675 - INFO - Pipeline started. Run ID: ad6d6980-3c66-4c43-bbf5-a98fdd2acce8
2026-11-06 13:01:05,221 - INFO - Pipeline completed successfully. Run ID: ad6d6980-3c66-4c43-bbf5-a98fdd2acce8
2026-11-06 13:01:39,244 - ERROR - Pipeline failed. Run ID: ad6d6980-3c66-4c43-bbf5-a98fdd2acce8. Reason: Simulated data processing failure
2026-11-06 13:22:52,610 - INFO - Pipeline started. Run ID: 7a5106b0-7686-4993-86b9-36d701ee36fa
2026-11-06 13:22:53,667 - INFO - Pipeline completed successfully. Run ID: 7a5106b0-7686-4993-86b9-36d701ee36fa



### Test failure logging

In [71]:
try:
    # Simulate a pipeline failure
    raise ValueError("Simulated data processing failure")

except Exception as e:
    failure_reason = str(e)

    logging.error(
        f"Pipeline failed. Run ID: {run_id}. "
        f"Reason: {failure_reason}"
    )

    print("Failure captured:", failure_reason)

Failure captured: Simulated data processing failure


# Step 10 – Automated testing

## What are we going to test?

### Create testing section

In [72]:
import pytest
import pandas as pd
import numpy as np

print("Testing libraries imported successfully!")

Testing libraries imported successfully!


### Create a reusable validation function

In [73]:
def validate_order_items(df, valid_order_ids, valid_product_ids):

    required_columns = [
        "order_item_id",
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount"
    ]

    # Check schema
    missing_columns = set(required_columns) - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    data = df.copy()

    # Store rejection reasons
    data["rejection_reason"] = ""

    def add_reason(mask, reason):
        data.loc[mask, "rejection_reason"] = (
            data.loc[mask, "rejection_reason"]
            .apply(lambda x: f"{x}; {reason}" if x else reason)
        )

    # Null checks
    for column in required_columns:
        add_reason(
            data[column].isna(),
            f"missing_{column}"
        )

    # Duplicate check
    duplicate_mask = (
        data["order_item_id"].duplicated(keep="first")
        & data["order_item_id"].notna()
    )

    add_reason(duplicate_mask, "duplicate_order_item_id")

    # Referential integrity
    add_reason(
        ~data["order_id"].isin(valid_order_ids),
        "invalid_order_id"
    )

    add_reason(
        ~data["product_id"].isin(valid_product_ids),
        "invalid_product_id"
    )

    # Business rules
    add_reason(
        pd.to_numeric(data["quantity"], errors="coerce").le(0),
        "invalid_quantity"
    )

    add_reason(
        pd.to_numeric(data["unit_price"], errors="coerce").le(0),
        "invalid_unit_price"
    )

    add_reason(
        pd.to_numeric(data["discount"], errors="coerce").lt(0),
        "negative_discount"
    )

    add_reason(
        pd.to_numeric(data["discount"], errors="coerce")
        > (
            pd.to_numeric(data["quantity"], errors="coerce")
            * pd.to_numeric(data["unit_price"], errors="coerce")
        ),
        "discount_exceeds_gross_amount"
    )

    valid = data[data["rejection_reason"] == ""].drop(
        columns="rejection_reason"
    )

    rejected = data[data["rejection_reason"] != ""]

    return valid.copy(), rejected.copy()

### Save the validation function as a Python module

In [74]:
os.makedirs("src", exist_ok=True)
os.makedirs("tests", exist_ok=True)

print("Testing folders ready!")

Testing folders ready!


In [75]:
%%writefile src/validation.py

import pandas as pd


def validate_order_items(df, valid_order_ids, valid_product_ids):

    required_columns = [
        "order_item_id", "order_id", "product_id",
        "quantity", "unit_price", "discount"
    ]

    missing = set(required_columns) - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    data = df.copy()
    data["rejection_reason"] = ""

    def add_reason(mask, reason):
        data.loc[mask, "rejection_reason"] = (
            data.loc[mask, "rejection_reason"].apply(
                lambda x: f"{x}; {reason}" if x else reason
            )
        )

    for column in required_columns:
        add_reason(data[column].isna(), f"missing_{column}")

    duplicate_mask = (
        data["order_item_id"].duplicated(keep="first")
        & data["order_item_id"].notna()
    )
    add_reason(duplicate_mask, "duplicate_order_item_id")

    add_reason(
        ~data["order_id"].isin(valid_order_ids),
        "invalid_order_id"
    )

    add_reason(
        ~data["product_id"].isin(valid_product_ids),
        "invalid_product_id"
    )

    for column in ["quantity", "unit_price", "discount"]:
        data[column] = pd.to_numeric(
            data[column], errors="coerce"
        )

    for column in ["quantity", "unit_price", "discount"]:
        add_reason(
            data[column].isna(),
            f"invalid_{column}"
        )

    add_reason(data["quantity"].le(0), "invalid_quantity")
    add_reason(data["unit_price"].le(0), "invalid_unit_price")
    add_reason(data["discount"].lt(0), "negative_discount")

    gross = data["quantity"] * data["unit_price"]

    add_reason(
        data["discount"].gt(gross),
        "discount_exceeds_gross_amount"
    )

    valid = data[data["rejection_reason"] == ""].drop(
        columns="rejection_reason"
    )

    rejected = data[data["rejection_reason"] != ""]

    return valid.copy(), rejected.copy()

Overwriting src/validation.py


### Create automated test cases

In [76]:
%%writefile tests/test_pipeline.py

import pandas as pd
import pytest

from src.validation import validate_order_items


def sample_data():
    return pd.DataFrame([{
        "order_item_id": "ITEM001",
        "order_id": "ORD001",
        "product_id": "PROD001",
        "quantity": 2,
        "unit_price": 100,
        "discount": 10
    }])


def test_happy_path():
    valid, rejected = validate_order_items(
        sample_data(), ["ORD001"], ["PROD001"]
    )
    assert len(valid) == 1
    assert len(rejected) == 0


def test_empty_input():
    empty = sample_data().iloc[0:0]
    valid, rejected = validate_order_items(
        empty, ["ORD001"], ["PROD001"]
    )
    assert len(valid) == 0
    assert len(rejected) == 0


def test_missing_required_column():
    data = sample_data().drop(columns=["product_id"])
    with pytest.raises(ValueError):
        validate_order_items(data, ["ORD001"], ["PROD001"])


def test_duplicate_records():
    data = pd.concat(
        [sample_data(), sample_data()],
        ignore_index=True
    )
    valid, rejected = validate_order_items(
        data, ["ORD001"], ["PROD001"]
    )
    assert len(valid) == 1
    assert len(rejected) == 1


def test_null_values():
    data = sample_data()
    data.loc[0, "product_id"] = None

    valid, rejected = validate_order_items(
        data, ["ORD001"], ["PROD001"]
    )
    assert len(rejected) == 1


def test_invalid_foreign_key():
    data = sample_data()
    data.loc[0, "product_id"] = "PROD999"

    valid, rejected = validate_order_items(
        data, ["ORD001"], ["PROD001"]
    )
    assert len(rejected) == 1


def test_negative_quantity():
    data = sample_data()
    data.loc[0, "quantity"] = -5

    valid, rejected = validate_order_items(
        data, ["ORD001"], ["PROD001"]
    )
    assert len(rejected) == 1


def test_late_arriving_record():
    data = sample_data()
    # Historical source events are not rejected just for being old.
    data["order_date"] = pd.Timestamp("2024-01-01")

    valid, rejected = validate_order_items(
        data, ["ORD001"], ["PROD001"]
    )
    assert len(valid) == 1
    assert len(rejected) == 0


def test_same_input_is_deterministic():
    data = sample_data()

    valid1, rejected1 = validate_order_items(
        data, ["ORD001"], ["PROD001"]
    )

    valid2, rejected2 = validate_order_items(
        data, ["ORD001"], ["PROD001"]
    )

    pd.testing.assert_frame_equal(valid1, valid2)
    pd.testing.assert_frame_equal(rejected1, rejected2)


def test_processing_failure_is_detected():
    data = sample_data()

    with pytest.raises(ValueError):
        validate_order_items(
            data.drop(columns=["order_id"]),
            ["ORD001"],
            ["PROD001"]
        )

Overwriting tests/test_pipeline.py


### Run automated tests

In [77]:
!python -m pytest -v tests

============================= test session starts =============================
platform win32 -- Python 3.13.11, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\Administrator\anaconda4\envs\shivaprasad741\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Administrator\ecommerce-data-warehouse
plugins: anyio-4.10.0, Faker-40.44.0
collecting ... collected 10 items

tests/test_pipeline.py::test_happy_path PASSED                           [ 10%]
tests/test_pipeline.py::test_empty_input PASSED                          [ 20%]
tests/test_pipeline.py::test_missing_required_column PASSED              [ 30%]
tests/test_pipeline.py::test_duplicate_records PASSED                    [ 40%]
tests/test_pipeline.py::test_null_values PASSED                          [ 50%]
tests/test_pipeline.py::test_invalid_foreign_key PASSED                  [ 60%]
tests/test_pipeline.py::test_negative_quantity PASSED                    [ 70%]
tests/test_pipeline.py::test_late_arriving_record PASSED                 [ 80%]

### Save test execution evidence

In [78]:
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "-v", "tests"],
    capture_output=True,
    text=True
)

print(result.stdout)

with open("output/test_results.txt", "w") as file:
    file.write(result.stdout)
    file.write(result.stderr)

print("Test execution evidence saved!")

============================= test session starts =============================
platform win32 -- Python 3.13.11, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\Administrator\anaconda4\envs\shivaprasad741\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Administrator\ecommerce-data-warehouse
plugins: anyio-4.10.0, Faker-40.44.0
collecting ... collected 10 items

tests/test_pipeline.py::test_happy_path PASSED                           [ 10%]
tests/test_pipeline.py::test_empty_input PASSED                          [ 20%]
tests/test_pipeline.py::test_missing_required_column PASSED              [ 30%]
tests/test_pipeline.py::test_duplicate_records PASSED                    [ 40%]
tests/test_pipeline.py::test_null_values PASSED                          [ 50%]
tests/test_pipeline.py::test_invalid_foreign_key PASSED                  [ 60%]
tests/test_pipeline.py::test_negative_quantity PASSED                    [ 70%]
tests/test_pipeline.py::test_late_arriving_record PASSED                 [ 80%]

# Step 11 – Idempotency and recovery

## Create an audit table

In [79]:
con.execute("""
CREATE TABLE IF NOT EXISTS pipeline_audit (
    run_id VARCHAR PRIMARY KEY,
    start_time VARCHAR,
    end_time VARCHAR,
    status VARCHAR,
    customer_count INTEGER,
    product_count INTEGER,
    date_count INTEGER,
    fact_count INTEGER,
    error_message VARCHAR
)
""")

print("Pipeline audit table created!")

Pipeline audit table created!


### Create an idempotent loading function

In [80]:
def load_warehouse_idempotent(
    con,
    dim_customer,
    dim_product,
    dim_date,
    fact_sales,
    simulate_failure=False
):
    run_id = str(uuid.uuid4())
    start_time = datetime.now(timezone.utc)

    try:
        con.execute("BEGIN TRANSACTION")

        # Delete existing records in dependency order
        con.execute("DELETE FROM fact_sales")
        con.execute("DELETE FROM dim_date")
        con.execute("DELETE FROM dim_product")
        con.execute("DELETE FROM dim_customer")

        # Register DataFrames
        con.register("customer_stage", dim_customer)
        con.register("product_stage", dim_product)
        con.register("date_stage", dim_date)
        con.register("fact_stage", fact_sales)

        # Load dimensions
        con.execute("""
            INSERT INTO dim_customer
            SELECT * FROM customer_stage
        """)

        con.execute("""
            INSERT INTO dim_product
            SELECT * FROM product_stage
        """)

        con.execute("""
            INSERT INTO dim_date
            SELECT * FROM date_stage
        """)

        # Simulate failure before loading facts
        if simulate_failure:
            raise RuntimeError("Simulated failure during fact loading")

        # Load fact table
        con.execute("""
            INSERT INTO fact_sales
            SELECT * FROM fact_stage
        """)

        end_time = datetime.now(timezone.utc)

        # Record successful execution
        con.execute("""
            INSERT INTO pipeline_audit VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, [
            run_id,
            start_time.isoformat(),
            end_time.isoformat(),
            "SUCCESS",
            len(dim_customer),
            len(dim_product),
            len(dim_date),
            len(fact_sales),
            None
        ])

        con.execute("COMMIT")

        print("Warehouse loaded successfully!")
        print("Run ID:", run_id)

    except Exception as e:

        con.execute("ROLLBACK")

        end_time = datetime.now(timezone.utc)

        # Record failure after rollback
        con.execute("""
            INSERT INTO pipeline_audit VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, [
            run_id,
            start_time.isoformat(),
            end_time.isoformat(),
            "FAILED",
            len(dim_customer),
            len(dim_product),
            len(dim_date),
            0,
            str(e)
        ])

        print("Pipeline failed:", str(e))
        print("Database transaction rolled back.")

    finally:
        # Remove registered temporary DataFrame references
        for name in [
            "customer_stage",
            "product_stage",
            "date_stage",
            "fact_stage"
        ]:
            try:
                con.unregister(name)
            except Exception:
                pass

    return run_id

### Execute the first warehouse load

In [81]:
# Clear fact table first
con.execute("DELETE FROM fact_sales")

# Clear dimension tables after fact table
con.execute("DELETE FROM dim_customer")
con.execute("DELETE FROM dim_product")
con.execute("DELETE FROM dim_date")

In [82]:
first_run_id = load_warehouse_idempotent(
    con,
    dim_customer,
    dim_product,
    dim_date,
    fact_sales
)

Warehouse loaded successfully!
Run ID: 834ce374-afa0-4fcc-a024-1b0c8e311e42


### Check database record counts

In [83]:
def get_warehouse_counts(con):

    tables = [
        "dim_customer",
        "dim_product",
        "dim_date",
        "fact_sales"
    ]

    counts = {}

    for table in tables:
        counts[table] = con.execute(
            f"SELECT COUNT(*) FROM {table}"
        ).fetchone()[0]

    return counts


first_counts = get_warehouse_counts(con)

print("First execution:")
print(first_counts)

First execution:
{'dim_customer': 98, 'dim_product': 29, 'dim_date': 132, 'fact_sales': 323}


In [84]:
# Clear fact table first
con.execute("DELETE FROM fact_sales")

# Clear dimension tables after fact table
con.execute("DELETE FROM dim_customer")
con.execute("DELETE FROM dim_product")
con.execute("DELETE FROM dim_date")

In [85]:
second_run_id = load_warehouse_idempotent(
    con,
    dim_customer,
    dim_product,
    dim_date,
    fact_sales
)

second_counts = get_warehouse_counts(con)

print("Second execution:")
print(second_counts)

assert first_counts == second_counts

print("IDEMPOTENCY TEST PASSED!")

Warehouse loaded successfully!
Run ID: 62472e53-c289-4c7e-9752-ac3aff93f082
Second execution:
{'dim_customer': 98, 'dim_product': 29, 'dim_date': 132, 'fact_sales': 323}
IDEMPOTENCY TEST PASSED!


### Test failure and rollback

In [86]:
# Clear fact table first
con.execute("DELETE FROM fact_sales")

# Clear dimension tables after fact table
con.execute("DELETE FROM dim_customer")
con.execute("DELETE FROM dim_product")
con.execute("DELETE FROM dim_date")

In [87]:
before_failure = get_warehouse_counts(con)

failed_run_id = load_warehouse_idempotent(
    con,
    dim_customer,
    dim_product,
    dim_date,
    fact_sales,
    simulate_failure=True
)

after_failure = get_warehouse_counts(con)

assert before_failure == after_failure

print("ROLLBACK TEST PASSED!")
print("Warehouse data remained unchanged after failure.")

Pipeline failed: Simulated failure during fact loading
Database transaction rolled back.
ROLLBACK TEST PASSED!
Warehouse data remained unchanged after failure.


In [88]:
retry_run_id = load_warehouse_idempotent(
    con,
    dim_customer,
    dim_product,
    dim_date,
    fact_sales
)

print("Retry completed.")
print(get_warehouse_counts(con))

Warehouse loaded successfully!
Run ID: 738bc82d-e676-4b44-8033-4cef563a395a
Retry completed.
{'dim_customer': 98, 'dim_product': 29, 'dim_date': 132, 'fact_sales': 323}


### Inspect pipeline audit history

In [89]:
audit_report = con.execute("""
SELECT *
FROM pipeline_audit
ORDER BY start_time
""").df()

display(audit_report)

,run_id,start_time,end_time,status,customer_count,product_count,date_count,fact_count,error_message
0,08fe90b5-fde5-4fa7-9822-9334bee1d659,2026-11-06T07:40:46.984243+00:00,2026-11-06T07:40:47.044343+00:00,FAILED,98,29,125,0,Constraint Error: Violates foreign key constra...
1,4537b6b3-330e-4a6a-adfa-a7523a80add8,2026-11-06T07:42:15.031750+00:00,2026-11-06T07:42:15.070386+00:00,SUCCESS,98,29,125,299,NaN
2,fc5a403b-06b2-4ba3-a361-f3d0ef7b0d43,2026-11-06T07:42:22.149213+00:00,2026-11-06T07:42:22.153889+00:00,FAILED,98,29,125,0,Constraint Error: Violates foreign key constra...
3,7af0119e-e29b-4708-9d84-ce831072e1be,2026-11-06T07:42:26.663946+00:00,2026-11-06T07:42:26.689785+00:00,SUCCESS,98,29,125,299,NaN
4,e6bb4090-4697-4687-9386-b455141ad70a,2026-11-06T07:43:13.468166+00:00,2026-11-06T07:43:13.472843+00:00,FAILED,98,29,125,0,Constraint Error: Violates foreign key constra...
5,cb0efc34-f198-4476-8e77-c266967f1aa8,2026-11-06T07:43:35.800281+00:00,2026-11-06T07:43:35.828050+00:00,SUCCESS,98,29,125,299,NaN
6,35ca28ef-0f08-4850-9b8d-e8174c441e40,2026-11-06T07:44:03.991287+00:00,2026-11-06T07:44:03.995079+00:00,FAILED,98,29,125,0,Constraint Error: Violates foreign key constra...
7,15d3b3ba-047b-4f44-98a4-fca2bd6070f4,2026-11-06T07:44:19.262146+00:00,2026-11-06T07:44:19.282724+00:00,FAILED,98,29,125,0,Simulated failure during fact loading
8,c3be2566-43df-492c-b5ec-7baa3e4ac022,2026-11-06T07:44:39.811097+00:00,2026-11-06T07:44:39.836298+00:00,SUCCESS,98,29,125,299,NaN
9,4639a7f7-87f6-4787-8270-342aa78b107f,2026-11-06T07:47:16.096298+00:00,2026-11-06T07:47:16.112568+00:00,FAILED,98,29,125,0,Resource In Use Error: connection has an unfin...


### Verify execution history

In [90]:
con.execute("""
SELECT
    status,
    COUNT(*) AS total_runs
FROM pipeline_audit
GROUP BY status
""").df()

,status,total_runs
0,FAILED,7
1,SUCCESS,7


In [91]:
audit_report.to_csv(
    "logs/pipeline_audit.csv",
    index=False
)

print("Pipeline audit report saved!")

Pipeline audit report saved!


In [92]:
print("Warehouse record counts:")

print(get_warehouse_counts(con))

print("\nPipeline execution history:")

display(
    con.execute("""
        SELECT run_id, status, error_message
        FROM pipeline_audit
        ORDER BY start_time DESC
    """).df()
)

Warehouse record counts:
{'dim_customer': 98, 'dim_product': 29, 'dim_date': 132, 'fact_sales': 323}

Pipeline execution history:


,run_id,status,error_message
0,738bc82d-e676-4b44-8033-4cef563a395a,SUCCESS,NaN
1,43fb73bc-0aa9-48ce-bda6-6fbb88e65699,FAILED,Simulated failure during fact loading
2,62472e53-c289-4c7e-9752-ac3aff93f082,SUCCESS,NaN
3,834ce374-afa0-4fcc-a024-1b0c8e311e42,SUCCESS,NaN
4,4639a7f7-87f6-4787-8270-342aa78b107f,FAILED,Resource In Use Error: connection has an unfin...
5,c3be2566-43df-492c-b5ec-7baa3e4ac022,SUCCESS,NaN
6,15d3b3ba-047b-4f44-98a4-fca2bd6070f4,FAILED,Simulated failure during fact loading
7,35ca28ef-0f08-4850-9b8d-e8174c441e40,FAILED,Constraint Error: Violates foreign key constra...
8,cb0efc34-f198-4476-8e77-c266967f1aa8,SUCCESS,NaN
9,e6bb4090-4697-4687-9386-b455141ad70a,FAILED,Constraint Error: Violates foreign key constra...


In [96]:
import duckdb

con = duckdb.connect(
    "output/ecommerce_warehouse.duckdb"
)

result = con.execute("""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT order_item_id) AS unique_records
FROM fact_sales
""").fetchdf()

display(result)

,total_records,unique_records
0,323,323


In [97]:
before_count = con.execute("""
SELECT COUNT(*) FROM fact_sales
""").fetchone()[0]

print("Before reload:", before_count)

Before reload: 323


In [98]:
try:
    con.execute("BEGIN TRANSACTION")

    con.execute("DELETE FROM fact_sales")

    con.register("fact_stage", fact_sales)

    con.execute("""
        INSERT INTO fact_sales
        SELECT * FROM fact_stage
    """)

    con.execute("COMMIT")

    print("Fact table reloaded successfully!")

except Exception as e:
    con.execute("ROLLBACK")
    print("Reload failed:", e)

finally:
    try:
        con.unregister("fact_stage")
    except Exception:
        pass

Fact table reloaded successfully!


In [103]:
after_count = con.execute("""
SELECT COUNT(*) FROM fact_sales
""").fetchone()[0]

print("Before:", before_count)
print("After:", after_count)

assert before_count == after_count
print("Idempotency verification passed!")

Before: 323
After: 323
Idempotency verification passed!


### Step 12 – Performance testing

In [104]:
import numpy as np
import pandas as pd
import time
import os

# Fixed random seed
rng = np.random.default_rng(42)

# Generate 100,000 records
n = 100_000

start_generation = time.perf_counter()

performance_items = pd.DataFrame({
    "order_item_id": [
        f"PERFITEM{i:07d}" for i in range(1, n + 1)
    ],
    "order_id": rng.choice(
        completed_orders["order_id"].to_numpy(),
        size=n
    ),
    "product_id": rng.choice(
        products_valid["product_id"].to_numpy(),
        size=n
    ),
    "quantity": rng.integers(1, 6, size=n)
})

# Look up prices from the product source
price_map = products_valid.set_index(
    "product_id"
)["unit_price"]

performance_items["unit_price"] = (
    performance_items["product_id"].map(price_map)
)

# Generate valid discounts
performance_items["discount"] = (
    rng.random(n) *
    performance_items["quantity"] *
    performance_items["unit_price"] * 0.20
).round(2)

generation_time = (
    time.perf_counter() - start_generation
)

print("Generated records:", len(performance_items))
print("Generation time:", round(generation_time, 3), "seconds")

display(performance_items.head())

Generated records: 100000
Generation time: 0.151 seconds


,order_item_id,order_id,product_id,quantity,unit_price,discount
0,PERFITEM0000001,ORD00050,PROD023,5,8398.92,7083.63
1,PERFITEM0000002,ORD00383,PROD027,5,13333.63,10055.01
2,PERFITEM0000003,ORD00333,PROD018,5,23185.14,11620.76
3,PERFITEM0000004,ORD00225,PROD014,5,40049.57,24062.04
4,PERFITEM0000005,ORD00224,PROD002,5,42995.81,31547.83


### Check dataset quality

In [105]:
assert len(performance_items) == 100_000

assert performance_items["order_item_id"].is_unique

assert performance_items["order_id"].isin(
    completed_orders["order_id"]
).all()

assert performance_items["product_id"].isin(
    products_valid["product_id"]
).all()

assert (performance_items["quantity"] > 0).all()

assert (
    performance_items["discount"] <=
    performance_items["quantity"] *
    performance_items["unit_price"]
).all()

print("100,000-record dataset validation passed!")

100,000-record dataset validation passed!


### Measure transformation performance

In [106]:
start_transform = time.perf_counter()

# Join order information
performance_sales = performance_items.merge(
    completed_orders[
        ["order_id", "customer_id", "order_date"]
    ],
    on="order_id",
    how="inner",
    validate="many_to_one"
)

# Join surrogate keys
performance_sales = performance_sales.merge(
    dim_customer[["customer_id", "customer_key"]],
    on="customer_id",
    how="inner",
    validate="many_to_one"
)

performance_sales = performance_sales.merge(
    dim_product[["product_id", "product_key"]],
    on="product_id",
    how="inner",
    validate="many_to_one"
)

# Generate date key
performance_sales["date_key"] = (
    pd.to_datetime(performance_sales["order_date"])
    .dt.strftime("%Y%m%d")
    .astype(int)
)

# Vectorized calculations
performance_sales["gross_amount"] = (
    performance_sales["quantity"] *
    performance_sales["unit_price"]
)

performance_sales["sales_amount"] = (
    performance_sales["gross_amount"] -
    performance_sales["discount"]
)

transform_time = (
    time.perf_counter() - start_transform
)

print("Transformed records:", len(performance_sales))
print("Transformation time:", round(transform_time, 3), "seconds")

Transformed records: 100000
Transformation time: 2.317 seconds


### Measure one optimization

In [111]:
benchmark_sample = performance_items.head(20_000).copy()

# Row-by-row calculation
start = time.perf_counter()

rowwise_amount = benchmark_sample.apply(
    lambda row:
        row["quantity"] * row["unit_price"] - row["discount"],
    axis=1
)

rowwise_time = time.perf_counter() - start

# Vectorized calculation
start = time.perf_counter()

vectorized_amount = (
    benchmark_sample["quantity"] *
    benchmark_sample["unit_price"] -
    benchmark_sample["discount"]
)

vectorized_time = time.perf_counter() - start

assert np.allclose(
    rowwise_amount.to_numpy(),
    vectorized_amount.to_numpy()
)

print("Row-wise time:", round(rowwise_time, 5), "seconds")
print("Vectorized time:", round(vectorized_time, 5), "seconds")

if vectorized_time > 0:
    print(
        "Speed improvement:",
        round(rowwise_time / vectorized_time, 2),
        "times"
    )

print("Both calculation methods produced matching results!")

Row-wise time: 0.48519 seconds
Vectorized time: 0.00118 seconds
Speed improvement: 411.98 times
Both calculation methods produced matching results!


### Generate performance report

In [116]:
import duckdb
import time

# Create an independent in-memory database
perf_con = duckdb.connect(":memory:")

start_load = time.time()

perf_con.register("performance_sales_df", performance_sales)

perf_con.execute("""
    CREATE TABLE performance_fact AS
    SELECT * FROM performance_sales_df
""")

loaded_count = perf_con.execute(
    "SELECT COUNT(*) FROM performance_fact"
).fetchone()[0]

load_time = time.time() - start_load

print("Records loaded:", loaded_count)
print("DuckDB load time:", round(load_time, 4), "seconds")

perf_con.close()

Records loaded: 100000
DuckDB load time: 0.1125 seconds


### Verify final performance output

In [117]:
assert loaded_count == 100_000

assert performance_sales["order_item_id"].is_unique

print("PERFORMANCE TEST PASSED!")
print("100,000 records processed and loaded successfully.")

PERFORMANCE TEST PASSED!
100,000 records processed and loaded successfully.


### Step 13: Final documentation and project preparatio

In [118]:
import os

for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in [
        ".ipynb_checkpoints",
        "__pycache__",
        ".pytest_cache"
    ]]

    level = root.count(os.sep)
    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

./
    Untitled.ipynb
    data/
        customers.csv
        orders.csv
        order_items.csv
        products.csv
        quarantine/
    logs/
        pipeline.log
        pipeline_audit.csv
        pipeline_run_summary.csv
    output/
        category_sales.csv
        dim_customer.csv
        dim_date.csv
        dim_product.csv
        ecommerce_warehouse.duckdb
        ecommerce_warehouse.duckdb.wal
        fact_sales.csv
        ingestion_summary.csv
        monthly_sales.csv
        regional_sales.csv
        test_results.txt
        top_customers.csv
        top_products.csv
    sql/
    src/
        validation.py
    tests/
        test_pipeline.py


### README.md

In [120]:
from pathlib import Path

readme = """
# E-Commerce Data Warehouse & ETL Pipeline

## 1. Project Overview

This project builds a small dimensional data warehouse
for an e-commerce business using Python, Pandas, SQL
and DuckDB.

The pipeline ingests transactional CSV files, validates
records, separates rejected data, transforms valid data
into dimension and fact tables, and publishes the data
to DuckDB for analytical reporting.

## 2. Technologies Used

- Python
- Pandas
- NumPy
- DuckDB
- SQL
- Pytest
- Jupyter Notebook
- Git and GitHub

## 3. Architecture

CSV Source Data
    |
    v
Data Ingestion
    |
    v
Data Validation
    |
    +------> Rejected Records / Quarantine
    |
    v
Data Transformation
    |
    v
Dimensional Model
    |
    +------> dim_customer
    +------> dim_product
    +------> dim_date
    +------> fact_sales
    |
    v
DuckDB Warehouse
    |
    v
SQL Analytical Reports

## 4. Data Model

### Dimension Tables

- dim_customer: Customer information.
- dim_product: Product information.
- dim_date: Calendar information.

### Fact Table

- fact_sales: One row per order item belonging
  to a completed order.

### Fact Table Measures

- quantity
- unit_price
- discount
- gross_amount
- sales_amount

## 5. Data Quality Checks

The pipeline checks:

- Missing required values
- Duplicate records
- Invalid customer references
- Invalid product references
- Invalid order references
- Negative quantities
- Invalid prices
- Invalid discounts
- Invalid dates

Rejected records are separated from valid records.

## 6. Idempotency

The project includes a fact-table reload test.
Repeated loading of the same input should not increase
the number of fact records.

## 7. Performance Testing

The project includes a performance test using
100,000 synthetic order-item records.

It measures:

- Data generation time
- Transformation time
- DuckDB loading time
- Total processing time
- DataFrame memory usage

## 8. Testing

Automated tests are written using Pytest.

Run tests using:

    python -m pytest -v tests

## 9. Setup

Install the required libraries:

    pip install -r requirements.txt

Open the Jupyter Notebook:

    jupyter notebook

Run the notebook cells in sequence.

## 10. Project Outputs

- Dimensional CSV files
- DuckDB warehouse
- SQL analytical reports
- Pipeline logs
- Validation reports
- Performance report

## 11. Limitations and Future Improvements

- Add incremental loading with persistent source
  watermarks.
- Implement full slowly changing dimensions.
- Add automated scheduling.
- Add cloud storage and orchestration.
- Improve retry and recovery handling.
"""

Path("README.md").write_text(readme, encoding="utf-8")

print("README.md created successfully!")

README.md created successfully!


### Create .gitignore

In [121]:
gitignore = """
__pycache__/
.ipynb_checkpoints/
.pytest_cache/
*.pyc
.env
venv/
.venv/
"""

Path(".gitignore").write_text(
    gitignore.strip() + "\n",
    encoding="utf-8"
)

print(".gitignore created!")

.gitignore created!


### Create data dictionary

In [122]:
data_dictionary = pd.DataFrame([
    ["dim_customer", "customer_key", "INTEGER", "Primary Key"],
    ["dim_customer", "customer_id", "VARCHAR", "Source Customer ID"],
    ["dim_customer", "customer_name", "VARCHAR", "Customer Name"],
    ["dim_customer", "email", "VARCHAR", "Customer Email"],
    ["dim_product", "product_key", "INTEGER", "Primary Key"],
    ["dim_product", "product_id", "VARCHAR", "Source Product ID"],
    ["dim_product", "product_name", "VARCHAR", "Product Name"],
    ["dim_product", "category", "VARCHAR", "Product Category"],
    ["dim_date", "date_key", "INTEGER", "Primary Key YYYYMMDD"],
    ["dim_date", "full_date", "DATE", "Calendar Date"],
    ["fact_sales", "order_item_id", "VARCHAR", "Primary Key"],
    ["fact_sales", "order_id", "VARCHAR", "Source Order ID"],
    ["fact_sales", "customer_key", "INTEGER", "Customer Foreign Key"],
    ["fact_sales", "product_key", "INTEGER", "Product Foreign Key"],
    ["fact_sales", "date_key", "INTEGER", "Date Foreign Key"],
    ["fact_sales", "quantity", "INTEGER", "Quantity Sold"],
    ["fact_sales", "unit_price", "DOUBLE", "Price Per Unit"],
    ["fact_sales", "discount", "DOUBLE", "Discount Amount"],
    ["fact_sales", "gross_amount", "DOUBLE", "Quantity x Unit Price"],
    ["fact_sales", "sales_amount", "DOUBLE", "Revenue After Discount"]
], columns=[
    "table_name",
    "column_name",
    "data_type",
    "description"
])

display(data_dictionary)

data_dictionary.to_csv(
    "data_dictionary.csv",
    index=False
)

print("Data dictionary created!")

,table_name,column_name,data_type,description
0,dim_customer,customer_key,INTEGER,Primary Key
1,dim_customer,customer_id,VARCHAR,Source Customer ID
2,dim_customer,customer_name,VARCHAR,Customer Name
3,dim_customer,email,VARCHAR,Customer Email
4,dim_product,product_key,INTEGER,Primary Key
5,dim_product,product_id,VARCHAR,Source Product ID
6,dim_product,product_name,VARCHAR,Product Name
7,dim_product,category,VARCHAR,Product Category
8,dim_date,date_key,INTEGER,Primary Key YYYYMMDD
9,dim_date,full_date,DATE,Calendar Date


Data dictionary created!


### Verify the files

In [129]:
required_files = [
    "README.md",
    "requirements.txt",
    ".gitignore",
    "data_dictionary.csv",
    "ecommerce_data_warehouse.ipynb",
    "src/validation.py",
    "tests/test_pipeline.py",
    "output/performance_report.csv"
]

for file in required_files:
    print(
        file,
        " - ",
        "FOUND" if Path(file).exists() else "MISSING"
    )

README.md  -  FOUND
requirements.txt  -  FOUND
.gitignore  -  FOUND
data_dictionary.csv  -  FOUND
ecommerce_data_warehouse.ipynb  -  MISSING
src/validation.py  -  FOUND
tests/test_pipeline.py  -  FOUND
output/performance_report.csv  -  MISSING


In [124]:
from pathlib import Path
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders here:")
for item in Path(".").iterdir():
    print(item)

Current working directory:
C:\Users\Administrator\ecommerce-data-warehouse

Files and folders here:
.gitignore
.ipynb_checkpoints
.pytest_cache
data
data_dictionary.csv
logs
output
README.md
sql
src
tests
Untitled.ipynb


In [126]:
requirements = """
pandas
numpy
duckdb
faker
pytest
jupyter
"""

Path("requirements.txt").write_text(
    requirements.strip() + "\n",
    encoding="utf-8"
)

print("requirements.txt created:", Path("requirements.txt").exists())

requirements.txt created: True


In [127]:
notebooks = list(Path(".").rglob("*.ipynb"))

for notebook in notebooks:
    print(notebook)

Untitled.ipynb
.ipynb_checkpoints\Untitled-checkpoint.ipynb


In [130]:
reports = list(Path(".").rglob("performance_report.csv"))

for report in reports:
    print(report)